# **Speed calculation (Romain)**

Import the library used in this notebook + set the path

In [40]:
import cv2
import time
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
# DATA_DIR doit maintenant pointer vers data_processed
DATA_DIR = (PROJECT_ROOT / "data_processed").resolve()
# où chercher les séquences (2 dossiers différents)
SEQ_DIRS = [
    DATA_DIR / "train" / "images",
    DATA_DIR / "test"  / "images",
]

VIDEO_OUTPUT_DIR = PROJECT_ROOT / "data_processed" / "videos"
VIDEO_OUTPUT_DIR=VIDEO_OUTPUT_DIR.resolve()
TRAIN_ANNOTATIONS_DIR = PROJECT_ROOT / "data" / "DETRAC-Train-Annotations-XML" / "DETRAC-Train-Annotations-XML"
TRAIN_ANNOTATIONS_DIR = TRAIN_ANNOTATIONS_DIR.resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("Exists:", DATA_DIR.exists())
print("VIDEO_OUTPUT_DIR:", VIDEO_OUTPUT_DIR)

FPS = 25 # Frames per second for video playback
MAX_FRAMES = 2000  # Maximum number of frames to read/play just in case

PROJECT_ROOT: e:\ICV-Project
DATA_DIR: E:\ICV-Project\data_processed
Exists: True
VIDEO_OUTPUT_DIR: E:\ICV-Project\data_processed\videos


Reading of the frames of all the videos

In [21]:
# collecte des séquences MVI_* dans les 2 emplacements
sequences = []
for root in SEQ_DIRS:
    if root.exists():
        sequences.extend([p for p in root.iterdir() if p.is_dir() and p.name.startswith("MVI_")])

sequences = sorted(sequences, key=lambda p: (p.name, str(p.parent)))  # tri stable
print("Nb sequences:", len(sequences))
if not sequences:
    raise FileNotFoundError(f"Aucune sequence MVI_* dans {SEQ_DIRS}")

print("Example:", sequences[0])  # ex: .../train_images/images/MVI_20011

# lecture des images de la première séquence trouvée
imgs = sorted(sequences[0].glob("img*.jpg"))[:500]
frames = []
for p in imgs:
    im = cv2.imread(str(p))
    if im is None:
        print("WARN: impossible de lire", p)
        continue
    frames.append(im)  # préchargement


Nb sequences: 100
Example: E:\ICV-Project\data_processed\train\images\MVI_20011


Display the video with bonding boxes (*Deprecated - do not use*)

In [ ]:
# # Read video
# video_name = 'cctv052x2004080516x01638.avi'  # video file name
# video_path = os.path.join(VIDEO_DIR, video_name)  # build full path

# video = cv2.VideoCapture(video_path)  # open video file

# # Visualize video
# while True:
#     ret, frame = video.read()  # read one frame
#     if not ret:                # stop if no frame is returned (end of video)
#         break
    
#     cv2.imshow('frame', frame)  # display current frame

#     for (x, y, w, h) in face_rects:                         # Loop over detections (for each face)
#     cv2.rectangle(face_img, (x, y), (x+w, y+h),
#                     (255, 29, 0), 5)                     # Draw a rectangle around each vehicles



#     key = cv2.waitKey(5)              # wait ~5 ms
#     if key == ord('q'):                     # stop if 'q' key is pressed
#         break
        

# video.release()                 # release video object
# cv2.destroyAllWindows()         # close all OpenCV windows

Display the images with bonding boxes (*Deprecated - do not use*)

In [ ]:
# import os
# from PIL import Image, ImageDraw, ImageFont

# IMG_DIR = "data/vd8c/images"
# LBL_DIR = "data/vd8c/labels"

# img_path = os.path.join(IMG_DIR, fname)
# label_name = os.path.splitext(fname)[0] + ".txt"
# txt_path = os.path.join(LBL_DIR, label_name)

# img = Image.open(img_path).convert("RGB")
# W, H = img.size
# draw = ImageDraw.Draw(img)

# with open(txt_path, "r", encoding="utf-8") as f:
#     for line in f:
#         parts = line.strip().split()
#         if len(parts) != 5:
#             continue

#         class_id_str, x, y, w, h = parts
#         x, y, w, h = map(float, (x, y, w, h))

#         # YOLO -> coins en pixels
#         x_min = (x - w / 2) * W
#         y_min = (y - h / 2) * H
#         x_max = (x + w / 2) * W
#         y_max = (y + h / 2) * H

#         # clamp dans l'image
#         x_min = max(0, min(W - 1, x_min))
#         y_min = max(0, min(H - 1, y_min))
#         x_max = max(0, min(W - 1, x_max))
#         y_max = max(0, min(H - 1, y_max))

#         # rectangle
#         draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)

#         # texte = class_id (sans dictionnaire)
#         label = class_id_str

#         # position du texte (au-dessus si possible)
#         tx, ty = x_min, max(0, y_min - 18)

#         # fond du texte pour lisibilité
#         text_bbox = draw.textbbox((tx, ty), label, font=font)
#         draw.rectangle(text_bbox, fill="red")
#         draw.text((tx, ty), label, fill="white", font=font)

# img.show()


## Rebuild the video (CHANGE HERE TO SELECT THE VIDEO SEQUENCE TO REBUILD)

In [ ]:
def write_avi(
    frames,
    output_path,
    fps=25,
    codec="XVID"
):
    """
    frames : list[np.ndarray] (BGR OpenCV)
    output_path : str ou Path, ex: "output.avi"
    fps : int
    codec : FOURCC, ex: XVID, MJPG
    """

    assert len(frames) > 0, "Liste de frames vide"

    h, w = frames[0].shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*codec)
    writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        fps,
        (w, h)
    )

    if not writer.isOpened():
        raise RuntimeError("Impossible d'ouvrir le VideoWriter")

    for i, frame in enumerate(frames):
        if frame is None:
            raise RuntimeError(f"Frame {i} invalide")
        writer.write(frame)

    writer.release()
    print(f"Vidéo écrite : {output_path}")

Functions for writing and play the video

In [ ]:
# # Please ignore the commented code below if you don't need it (video rebuilt) - Ctrl+/ to uncomment

# i=0
# n=len(sequences)

# for seq in sequences:
#     video_name = seq.name
#     out = Path(VIDEO_OUTPUT_DIR, f"{video_name}.avi")

#     # --- création vidéo si absente ---
#     if not out.exists():
#         print(f"[WRITE] {video_name}")

#         imgs = sorted(seq.glob("img*.jpg"))
#         if MAX_FRAMES is not None:
#             imgs = imgs[:MAX_FRAMES]

#         if len(imgs) == 0:
#             print(f"[SKIP] aucune image pour {video_name}")
#             continue

#         frames = []
#         for p in imgs:
#             img = cv2.imread(str(p))
#             if img is None:
#                 raise RuntimeError(f"Image illisible : {p}")
#             frames.append(img)

#         write_avi(
#             frames=frames,
#             output_path=out,
#             fps=FPS,
#             codec="XVID"
#         )

#     #else:
#         #print(f"[EXISTS] {video_name}")

#     print(video_name,'rebuilt.',"Progress :", i+1, "/",n)
#     i+=1


## Play a video

In [ ]:
def play_video(video_path, window_name="Video", fps=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Impossible d’ouvrir la vidéo : {video_path}")

    # FPS depuis le fichier si non imposé
    if fps is None:
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            fps = 25  # fallback

    delay_ms = int(1000 / fps)

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            cv2.imshow(window_name, frame)

            key = cv2.waitKey(delay_ms) & 0xFF
            if key == ord('q') or key == 27:  # q ou ESC
                break

            # fermeture via la croix
            if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                break
    finally:
        cap.release()
        cv2.destroyWindow(window_name)
        cv2.waitKey(1)

In [45]:
video_name = "MVI_20011.avi"

video_path=VIDEO_OUTPUT_DIR / video_name
play_video(video_path, window_name=video_name)

## Calculation of vehicles speed

### Calibration (px to m)

In [ ]:
points = [(693, 338), (622, 175)]

In [ ]:
# import cv2
# from pathlib import Path

# seq = "MVI_20011"
# img_path = (PROJECT_ROOT / "data_processed" / "train" / "images" / seq / "img00001.jpg")

# img0 = cv2.imread(str(img_path))
# if img0 is None:
#     raise FileNotFoundError(f"Impossible de lire {img_path}")

# win = "Measure (move mouse, left-click to record, ESC to quit)"
# points = []
# x_cur, y_cur = 0, 0

# def on_mouse(event, x, y, flags, param):
#     global x_cur, y_cur
#     x_cur, y_cur = x, y
#     if event == cv2.EVENT_LBUTTONDOWN:
#         points.append((x, y))
#         print(f"Click -> x={x}, y={y}")

# cv2.namedWindow(win, cv2.WINDOW_NORMAL)
# cv2.setMouseCallback(win, on_mouse)

# while True:
#     img = img0.copy()

#     # crosshair
#     cv2.line(img, (0, y_cur), (img.shape[1], y_cur), (0, 255, 0), 1)
#     cv2.line(img, (x_cur, 0), (x_cur, img.shape[0]), (0, 255, 0), 1)

#     # text coords
#     cv2.putText(img, f"x={x_cur}, y={y_cur}", (10, 30),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

#     # draw saved points
#     for (xp, yp) in points:
#         cv2.circle(img, (xp, yp), 4, (0, 0, 255), -1)

#     cv2.imshow(win, img)
#     key = cv2.waitKey(10) & 0xFF
#     if key == 27:  # ESC
#         break

# cv2.destroyAllWindows()
# print("Points enregistres:", points)


Click -> x=693, y=338
Click -> x=622, y=175
Points enregistres: [(693, 338), (622, 175)]


In [31]:
(p1x, p1y), (p2x, p2y) = points
D_px = np.hypot(p2x - p1x, p2y - p1y)
D_m = 3.5  # metres (ex: largeur de voie)
meters_per_pixel = D_m / D_px
print("meters_per_pixel =", meters_per_pixel)

meters_per_pixel = 0.019685923651258114


### Compute the vehicles speed

In [46]:
import cv2
import numpy as np
from pathlib import Path

# =========================
# CONFIG
# =========================
FPS = 25  # frames per second


# =========================
# IO: read ONE vehicle from extended label file
# =========================
def read_vehicle_det_from_txt(txt_path: Path, vehicle_id: int):
    """
    Lit un fichier labels_xml/imgXXXXX.txt et retourne la detection correspondant a vehicle_id.

    Format attendu par ligne :
    class xc yc w h id orientation speed trajectory_length truncation_ratio vehicle_type density weather

    Retour :
      (cls, xc, yc, w, h) ou None si l'ID n'est pas present
    """
    # txt_path est issu d'un glob => il existe, mais on garde la securite
    if not txt_path.exists():
        return None

    # Lecture du fichier une seule fois
    for line in txt_path.read_text().splitlines():
        if not line.strip():
            continue

        parts = line.split()
        if len(parts) < 6:
            continue

        cls = int(float(parts[0]))
        xc  = float(parts[1])
        yc  = float(parts[2])
        w   = float(parts[3])
        h   = float(parts[4])
        vid = int(float(parts[5]))

        if vid == vehicle_id:
            return cls, xc, yc, w, h

    return None


# =========================
# Geometry: tracking point
# =========================
def bottom_center_px(det, W, H):
    """
    Point de suivi = centre bas de la bbox (pixel)
    """
    _, xc, yc, w, h = det
    x = xc * W
    y = (yc + 0.5 * h) * H
    return np.array([x, y], dtype=float)


# =========================
# Speed computation for ONE vehicle (TXT-ONLY)
# =========================
def compute_speed_for_vehicle(labels_xml_dir: Path,
                              vehicle_id: int,
                              image_size=None,
                              meters_per_pixel=None,
                              smooth_window=7):
    """
    Calcule la vitesse instantanee pour UN vehicule (ID donne) en parcourant UNIQUEMENT les .txt.

    Inputs:
      labels_xml_dir: .../labels_xml/<sequence> contenant imgXXXXX.txt
      vehicle_id: ID DETRAC a suivre
      image_size: (W,H) en pixels. Si None, on lit la premiere image du split pour l'obtenir (optionnel).
                 Recommande: passer (W,H) pour eviter toute lecture image.
      meters_per_pixel: float ou None
      smooth_window: lissage moyenne glissante (en nb d'echantillons vitesse)

    Output:
      dict avec v_px_s (+ v_m_s, v_km_h si meters_per_pixel fourni)
    """
    txt_paths = sorted(labels_xml_dir.glob("img*.txt"))
    if not txt_paths:
        raise FileNotFoundError(f"Aucun fichier img*.txt dans {labels_xml_dir}")

    # Taille image
    # - Si tu fournis image_size, on ne lit aucune image.
    # - Sinon, on ne peut pas convertir en pixels physiques (xc,yc sont normalises).
    if image_size is None:
        raise ValueError("image_size=(W,H) requis pour convertir xc,yc,w,h en pixels sans lire d'images.")
    W, H = image_size

    positions = []
    times = []

    for txt_path in txt_paths:
        # Temps depuis le nom de frame (img00088 -> 88). On reste coherent avec DETRAC:
        # frame num N correspond a N/FPS.
        frame_num = int(txt_path.stem.replace("img", ""))
        det = read_vehicle_det_from_txt(txt_path, vehicle_id)
        if det is None:
            continue

        p = bottom_center_px(det, W, H)
        positions.append(p)
        times.append(frame_num / FPS)

    if len(positions) < 2:
        raise RuntimeError(f"Pas assez de frames pour le vehicule ID={vehicle_id}")

    positions = np.array(positions)
    times = np.array(times)

    # Deplacements entre positions consecutives (pixels)
    d = np.linalg.norm(positions[1:] - positions[:-1], axis=1)

    # dt variable possible (si le vehicule disparait puis reapparait)
    dt = times[1:] - times[:-1]
    if np.any(dt <= 0):
        raise RuntimeError("Probleme d'ordre temporel: dt <= 0 detecte (noms de frames non ordonnes ?).")

    # vitesse px/s
    v_px_s = d / dt

    # Lissage simple
    if smooth_window and smooth_window > 1 and len(v_px_s) >= smooth_window:
        k = np.ones(smooth_window) / smooth_window
        v_px_s = np.convolve(v_px_s, k, mode="same")

    out = {
        "t": times[1:],
        "v_px_s": v_px_s
    }

    if meters_per_pixel is not None:
        v_m_s = v_px_s * meters_per_pixel
        out["v_m_s"] = v_m_s
        out["v_km_h"] = v_m_s * 3.6

    return out


# =========================
# Average speed (summary)
# =========================
def average_speed_for_vehicle(labels_xml_dir: Path,
                              vehicle_id: int,
                              image_size,
                              meters_per_pixel=None,
                              smooth_window=7):
    """
    Vitesse moyenne pour un vehicule donne (TXT-only)
    """
    res = compute_speed_for_vehicle(
        labels_xml_dir=labels_xml_dir,
        vehicle_id=vehicle_id,
        image_size=image_size,
        meters_per_pixel=meters_per_pixel,
        smooth_window=smooth_window
    )

    summary = {
        "vehicle_id": vehicle_id,
        "n_samples": int(len(res["v_px_s"])),
        "v_px_s_mean": float(np.mean(res["v_px_s"])),
        "v_px_s_median": float(np.median(res["v_px_s"])),
    }

    if "v_km_h" in res:
        summary["v_km_h_mean"] = float(np.mean(res["v_km_h"]))
        summary["v_km_h_median"] = float(np.median(res["v_km_h"]))

    return summary, res


# =========================
# EXEMPLE D'UTILISATION
# =========================
# seq = "MVI_20011"
# labels_xml_dir = DATA_DIR / "train" / "labels_xml" / seq
# image_size = (960, 540)  # A REMPLACER par la vraie taille de tes images (W,H)
# meters_per_pixel = 0.0246  # optionnel
# vehicle_id = 3
#
# summary, series = average_speed_for_vehicle(
#     labels_xml_dir=labels_xml_dir,
#     vehicle_id=vehicle_id,
#     image_size=image_size,
#     meters_per_pixel=meters_per_pixel,
#     smooth_window=7
# )
# print(summary)


### Example

In [48]:
image_size = (960, 540)   # (W, H) à adapter
seq = "MVI_20011"
vehicle_id = 3  # ID DETRAC a analyser

labels_xml_dir = DATA_DIR / "train" / "labels_xml" / seq

summary, series = average_speed_for_vehicle(
    labels_xml_dir=labels_xml_dir,
    vehicle_id=vehicle_id,
    image_size=image_size,
    meters_per_pixel=meters_per_pixel,
    smooth_window=7
)

print("=== VITESSE MOYENNE ===")
for k, v in summary.items():
    print(k, ":", v)


=== VITESSE MOYENNE ===
vehicle_id : 3
n_samples : 104
v_px_s_mean : 104.94695887927418
v_px_s_median : 68.27802572031514
v_km_h_mean : 7.437520151744817
v_km_h_median : 4.838817604999543


## Extract XML Metada (YOLO+VEHICLE UNIQUE ID)

In [28]:
from pathlib import Path
import cv2
import xml.etree.ElementTree as ET

def already_done(out_dir: Path) -> bool:
    return out_dir.exists()



# =========================
# 1) LISTE DES SEQUENCES (train + test) + PRECHARGEMENT FRAMES (exemple)
# =========================
DATA_DIR = (PROJECT_ROOT / "data_processed").resolve()

SEQ_DIRS = [
    DATA_DIR / "train" / "images",
    DATA_DIR / "test"  / "images",
]

sequences = []
for root in SEQ_DIRS:
    if root.exists():
        sequences.extend([p for p in root.iterdir() if p.is_dir() and p.name.startswith("MVI_")])

sequences = sorted(sequences, key=lambda p: (p.name, str(p.parent)))

print("Nb sequences:", len(sequences))
if not sequences:
    raise FileNotFoundError(f"Aucune sequence MVI_* dans {SEQ_DIRS}")
print("Example:", sequences[0])

imgs = sorted(sequences[0].glob("img*.jpg"))[:500]

frames = []
for p in imgs:
    im = cv2.imread(str(p))
    if im is None:
        print("WARN: impossible de lire", p)
        continue
    frames.append(im)

seq_img_dir = sequences[0]  # .../train/images/MVI_20011
seq_name = seq_img_dir.name
split_root = seq_img_dir.parent.parent  # .../train  ou .../test
seq_lbl_dir = split_root / "labels" / seq_name
print("Labels dir:", seq_lbl_dir)

# =========================
# 2) EXPORT XML -> labels_xml (1 fichier txt par frame)
# =========================

# Ajuste selon tes classes YOLO
CLASS_MAP = {"car": 0, "bus": 1, "van": 2, "others": 2}

# Si tu n'as pas de meteo, laisse vide => "unknown"
WEATHER_BY_SEQ = {}  # ex: {"MVI_20011":"sunny"}

TRAIN_XML_DIR = (PROJECT_ROOT / "data" / "DETRAC-Train-Annotations-XML" / "DETRAC-Train-Annotations-XML").resolve()
TEST_XML_DIR  = (PROJECT_ROOT / "data" / "DETRAC-Test-Annotations-XML"  / "DETRAC-Test-Annotations-XML").resolve()

def xml_to_yolo_norm(left, top, width, height, W, H):
    xc = (left + width / 2.0) / W
    yc = (top + height / 2.0) / H
    w  = width / W
    h  = height / H
    return xc, yc, w, h

def frame_label_name(num: int):
    return f"img{num:05d}.txt"

def parse_and_export_xml(xml_path: Path, out_dir: Path, W: int, H: int,
                         weather: str = "unknown", class_map=None):
    """
    Ecrit pour chaque frame: out_dir/imgxxxxx.txt
    Chaque ligne = 1 target XML :
    class xc yc w h id orientation speed trajectory_length truncation_ratio vehicle_type density weather
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for fr in root.iter("frame"):
        num = int(fr.attrib["num"])
        density = int(float(fr.attrib.get("density", "0")))

        lines_out = []
        tl = fr.find("target_list")
        if tl is not None:
            for t in tl.iter("target"):
                tid = int(t.attrib["id"])
                box = t.find("box")
                attr = t.find("attribute")
                if box is None or attr is None:
                    continue

                left   = float(box.attrib["left"])
                top    = float(box.attrib["top"])
                width  = float(box.attrib["width"])
                height = float(box.attrib["height"])

                xc, yc, w, h = xml_to_yolo_norm(left, top, width, height, W, H)

                vehicle_type = attr.attrib.get("vehicle_type", "unknown")
                orientation  = attr.attrib.get("orientation", "nan")
                speed        = attr.attrib.get("speed", "nan")
                traj_len     = attr.attrib.get("trajectory_length", "nan")
                trunc        = attr.attrib.get("truncation_ratio", "nan")

                cls = -1 if class_map is None else class_map.get(vehicle_type, -1)

                lines_out.append(
                    f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f} "
                    f"{tid} {orientation} {speed} {traj_len} {trunc} {vehicle_type} "
                    f"{density} {weather}"
                )

        (out_dir / frame_label_name(num)).write_text("\n".join(lines_out) + ("\n" if lines_out else ""))

def export_labels_xml_for_sequence(seq_img_dir: Path):
    """
    seq_img_dir = .../data_processed/<split>/images/<sequence>
    Cree .../data_processed/<split>/labels_xml/<sequence>/imgxxxxx.txt
    """
    seq_name = seq_img_dir.name
    split_root = seq_img_dir.parent.parent  # .../train ou .../test
    split = split_root.name  # "train" ou "test"

    # XML selon split
    xml_dir = TRAIN_XML_DIR if split == "train" else TEST_XML_DIR
    xml_path = xml_dir / f"{seq_name}.xml"
    if not xml_path.exists():
        print(f"WARN: XML manquant: {xml_path} (skip {split}/{seq_name})")
        return

    # Taille image via 1ere frame
    imgs = sorted(seq_img_dir.glob("img*.jpg"))
    if not imgs:
        print(f"WARN: pas d'images: {seq_img_dir} (skip)")
        return
    im0 = cv2.imread(str(imgs[0]))
    if im0 is None:
        print(f"WARN: image illisible: {imgs[0]} (skip)")
        return
    H, W = im0.shape[:2]

    out_dir = split_root / "labels_xml" / seq_name

    if already_done(out_dir):
        print(f"SKIP: {split}/{seq_name} (labels_xml deja cree)")
        return

    weather = WEATHER_BY_SEQ.get(seq_name, "unknown")
    parse_and_export_xml(xml_path, out_dir, W, H, weather=weather, class_map=CLASS_MAP)
    print(f"OK: {split}/{seq_name} -> {out_dir}")


# =========================
# 3) LANCER POUR TOUTES LES SEQUENCES
# =========================
i=0
for seq_img_dir in sequences:
    export_labels_xml_for_sequence(seq_img_dir)
    i+=1
    print("Progress :", i, "/",len(sequences))


Nb sequences: 88
Example: E:\ICV-Project\data_processed\train\images\MVI_20011
Labels dir: E:\ICV-Project\data_processed\train\labels\MVI_20011
SKIP: train/MVI_20011 (labels_xml deja cree)
Progress : 1 / 88
SKIP: train/MVI_20033 (labels_xml deja cree)
Progress : 2 / 88
SKIP: train/MVI_20034 (labels_xml deja cree)
Progress : 3 / 88
SKIP: train/MVI_20035 (labels_xml deja cree)
Progress : 4 / 88
SKIP: train/MVI_20051 (labels_xml deja cree)
Progress : 5 / 88
SKIP: train/MVI_20052 (labels_xml deja cree)
Progress : 6 / 88
SKIP: train/MVI_20062 (labels_xml deja cree)
Progress : 7 / 88
SKIP: train/MVI_20063 (labels_xml deja cree)
Progress : 8 / 88
SKIP: train/MVI_20064 (labels_xml deja cree)
Progress : 9 / 88
SKIP: test/MVI_39031 (labels_xml deja cree)
Progress : 10 / 88
SKIP: test/MVI_39051 (labels_xml deja cree)
Progress : 11 / 88
SKIP: test/MVI_39211 (labels_xml deja cree)
Progress : 12 / 88
SKIP: test/MVI_39271 (labels_xml deja cree)
Progress : 13 / 88
SKIP: test/MVI_39311 (labels_xml deja

## Homographie

In [ ]:
import numpy as np
from pathlib import Path

FPS = 25

# =========================================================
# A) HOMOGRAPHY (image pixels -> world meters)
# =========================================================
def compute_homography(img_pts_px, world_pts_m):
    """
    img_pts_px: array-like shape (N,2) with N>=4, pixel coordinates (x,y)
    world_pts_m: array-like shape (N,2), corresponding real-world coordinates (X,Y) in meters

    Returns:
      H: 3x3 homography matrix mapping image->world
    """
    img = np.asarray(img_pts_px, dtype=np.float64)
    wrd = np.asarray(world_pts_m, dtype=np.float64)

    if img.shape != wrd.shape or img.shape[0] < 4 or img.shape[1] != 2:
        raise ValueError("Need N>=4 corresponding points with shape (N,2) for both img_pts_px and world_pts_m.")

    # Normalized DLT via OpenCV is easiest, but to keep it pure numpy-free of cv2,
    # we'd need implement DLT. Here we rely on OpenCV if available.
    import cv2
    H, mask = cv2.findHomography(img, wrd, method=0)
    if H is None:
        raise RuntimeError("Homography estimation failed. Check point quality and correspondences.")
    return H

def apply_homography(H, x_px, y_px):
    """
    Apply homography to a single pixel point (x,y).
    Returns (X,Y) in meters.
    """
    p = np.array([x_px, y_px, 1.0], dtype=np.float64)
    q = H @ p
    if q[2] == 0:
        raise RuntimeError("Invalid homography projection (division by zero).")
    X = q[0] / q[2]
    Y = q[1] / q[2]
    return float(X), float(Y)

# =========================================================
# B) READ ONE VEHICLE LINE (labels_xml) BY ID
# Format line:
# class xc yc w h id orientation speed trajectory_length truncation_ratio vehicle_type density weather
# =========================================================
def read_vehicle_det_from_txt(txt_path: Path, vehicle_id: int):
    """
    Returns (cls, xc, yc, w, h) for the given vehicle_id, or None if absent.
    xc,yc,w,h are normalized YOLO floats.
    """
    if not txt_path.exists():
        return None

    for line in txt_path.read_text().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) < 6:
            continue

        cls = int(float(parts[0]))
        xc  = float(parts[1])
        yc  = float(parts[2])
        w   = float(parts[3])
        h   = float(parts[4])
        vid = int(float(parts[5]))

        if vid == vehicle_id:
            return cls, xc, yc, w, h

    return None

def bottom_center_px(det, W, H):
    """
    Tracking point = bottom center of bbox in pixels.
    """
    _, xc, yc, w, h = det
    x = xc * W
    y = (yc + 0.5 * h) * H
    return float(x), float(y)

# =========================================================
# C) SPEED COMPUTATION IN WORLD METERS USING HOMOGRAPHY
# =========================================================
def compute_speed_for_vehicle_homography(labels_xml_dir: Path,
                                         vehicle_id: int,
                                         image_size,
                                         H_img_to_world,
                                         smooth_window=7):
    """
    TXT-only parsing + homography speed.

    Inputs:
      labels_xml_dir: .../labels_xml/<sequence>/img*.txt
      vehicle_id: DETRAC id to follow
      image_size: (W,H) pixels
      H_img_to_world: 3x3 homography mapping image(px)->world(m)
      smooth_window: moving average on speed samples (optional)

    Returns:
      dict with:
        t (s), v_m_s, v_km_h, XY (world positions)
    """
    W, H = image_size
    txt_paths = sorted(labels_xml_dir.glob("img*.txt"))
    if not txt_paths:
        raise FileNotFoundError(f"Aucun fichier img*.txt dans {labels_xml_dir}")

    times = []
    XY = []

    for txt_path in txt_paths:
        # time from frame index in filename img00088 -> 88/FPS
        frame_num = int(txt_path.stem.replace("img", ""))

        det = read_vehicle_det_from_txt(txt_path, vehicle_id)
        if det is None:
            continue

        x_px, y_px = bottom_center_px(det, W, H)
        X_m, Y_m = apply_homography(H_img_to_world, x_px, y_px)

        times.append(frame_num / FPS)
        XY.append((X_m, Y_m))

    if len(XY) < 2:
        raise RuntimeError(f"Pas assez de frames pour le vehicule ID={vehicle_id}")

    times = np.asarray(times, dtype=np.float64)
    XY = np.asarray(XY, dtype=np.float64)

    d = np.linalg.norm(XY[1:] - XY[:-1], axis=1)     # meters
    dt = times[1:] - times[:-1]                      # seconds
    if np.any(dt <= 0):
        raise RuntimeError("Probleme d'ordre temporel: dt <= 0 (fichiers non ordonnes ?).")

    v_m_s = d / dt

    # smoothing
    if smooth_window and smooth_window > 1 and len(v_m_s) >= smooth_window:
        k = np.ones(smooth_window) / smooth_window
        v_m_s = np.convolve(v_m_s, k, mode="same")

    return {
        "t": times[1:],
        "XY_m": XY,
        "v_m_s": v_m_s,
        "v_km_h": v_m_s * 3.6,
    }

def average_speed_for_vehicle_homography(labels_xml_dir: Path,
                                        vehicle_id: int,
                                        image_size,
                                        H_img_to_world,
                                        smooth_window=7):
    """
    Average speed summary in km/h (and m/s).
    """
    res = compute_speed_for_vehicle_homography(
        labels_xml_dir=labels_xml_dir,
        vehicle_id=vehicle_id,
        image_size=image_size,
        H_img_to_world=H_img_to_world,
        smooth_window=smooth_window
    )

    summary = {
        "vehicle_id": vehicle_id,
        "n_samples": int(len(res["v_m_s"])),
        "v_m_s_mean": float(np.mean(res["v_m_s"])),
        "v_m_s_median": float(np.median(res["v_m_s"])),
        "v_km_h_mean": float(np.mean(res["v_km_h"])),
        "v_km_h_median": float(np.median(res["v_km_h"])),
    }
    return summary, res


In [ ]:
# 4 points au sol dans l'image (pixels)
img_pts_px = [
    (320, 650),  # P1
    (520, 650),  # P2
    (700, 420),  # P3
    (250, 420),  # P4
]

# Coordonnées réelles correspondantes dans le plan route (mètres)
# Choisis un repère simple: P1=(0,0), P2=(L,0), etc.
world_pts_m = [
    (0.0, 0.0),   # P1
    (3.5, 0.0),   # P2 (ex: largeur 3.5m)
    (6.0, 20.0),  # P3
    (-2.0, 20.0), # P4
]


In [ ]:
H_img_to_world = compute_homography(img_pts_px, world_pts_m)


In [ ]:
seq = "MVI_20011"
vehicle_id = 3

labels_xml_dir = DATA_DIR / "train" / "labels_xml" / seq

# Taille image (W,H) — tu peux la mettre en dur si tu la connais
image_size = (960, 540)  # A ADAPTER

summary, series = average_speed_for_vehicle_homography(
    labels_xml_dir=labels_xml_dir,
    vehicle_id=vehicle_id,
    image_size=image_size,
    H_img_to_world=H_img_to_world,
    smooth_window=7
)

print("=== VITESSE MOYENNE (HOMOGRAPHIE) ===")
for k, v in summary.items():
    print(k, ":", v)
